In [ ]:
import os
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

import torch
from gsplat import rasterization

from depth_anything_3.api import DepthAnything3

In [ ]:
# ---- env ----
os.environ["HF_HOME"] = "/work/scratch/nmeurer/hf_cache"
os.environ["HF_HUB_CACHE"] = "/work/scratch/nmeurer/hf_cache/hub"
os.environ["XDG_CACHE_HOME"] = "/work/scratch/nmeurer/cache"

# ---- paths ----
DATA_ROOT = Path("/cluster/courses/cil/monocular-depth-estimation/train")

# ---- config ----
TEACHER_MODEL="DA3-GIANT-1.1"

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

In [ ]:
!ls $DATA_ROOT | head -n 6

In [ ]:
# Load pretrained model
model = DepthAnything3.from_pretrained(f"depth-anything/{TEACHER_MODEL}", cache_dir=os.environ["HF_HUB_CACHE"])
model = model.to(device).eval()

In [ ]:
def visualize_data_pair(rgb_path, depth_path):
    # Load rgb
    rgb = np.array(Image.open(rgb_path).convert("RGB"), dtype=np.float32) / 255.0
    
    # Load depth map
    depth = np.load(depth_path).astype(np.float32)
    depth_vis = np.where(depth > 0, depth, np.nan)

    # Plot
    fig = plt.figure()
    ax_rgb = fig.add_subplot(1, 2, 1)
    ax_depth = fig.add_subplot(1, 2, 2)

    ax_rgb.imshow(rgb)
    ax_rgb.axis("off")
    ax_rgb.set_ylabel("RGB", fontsize=9, labelpad=4)

    im = ax_depth.imshow(depth_vis, cmap="viridis")
    ax_depth.axis("off")
    ax_depth.set_ylabel("Depth", fontsize=9, labelpad=4)
    plt.colorbar(im, ax=ax_depth, fraction=0.046, pad=0.04)

    fig.suptitle(rgb_path.name.replace("_rgb.png", ""))

    plt.show()

In [ ]:
# Sample rgb image path and ground truth depth map from the train dataset
rgb_files = sorted(DATA_ROOT.glob("*_rgb.png"))
index = random.randint(0, len(rgb_files) - 1)
rgb_path = rgb_files[index]
gtd_path = Path(str(rgb_path).replace("_rgb.png", "_depth.npy"))

visualize_data_pair(rgb_path, gtd_path)

In [ ]:
# First pass to get predicted camera extrinsics
prediction = model.inference(
    image=[str(rgb_path)],
    process_res=560,
    use_ray_pose=True,
)

# Shapes (N=1 for single image):
# prediction.depth        : (1, H, W) float32
# prediction.conf         : (1, H, W) float32   confidence per pixel
# prediction.extrinsics   : (1, 3, 4) float32   world-to-camera [R | t]
# prediction.intrinsics   : (1, 3, 3) float32
# prediction.processed_images : (1, H, W, 3) uint8   the actually-processed image

H, W = prediction.depth.shape[1:]
extr_w2c = prediction.extrinsics[0]   # (3, 4)
K        = prediction.intrinsics[0]   # (3, 3)
depth    = prediction.depth[0]        # (H, W)

# Extract rotation (world to cam) and translation (world to cam)
R = extr_w2c[:, :3]
t = extr_w2c[:, 3]

# Invert rotation to get cam to world and recover camera center in world frame
R_c2w = R.T
c_w = - R.T @ t

# Extract the world-space axes of the camera (OpenCV convention)
right_w = R_c2w[:, 0]
down_w = R_c2w[:, 1]
forward_w = R_c2w[:, 2]
up_w = -down_w

In [ ]:
### Pick a look-at target to toe the camera pose in the novel views back to center
# Take median depth over confident outputs
conf = prediction.conf.squeeze(0)
conf_mask = (conf > np.percentile(conf, 20)) & np.isfinite(depth) & (depth > 0)
scene_depth = float(np.median(depth[conf_mask]))

# Fix the look-at target and use a small shift-magnitude to avoid major dis-occlussion artifacts
T = c_w + scene_depth * forward_w
delta = 0.05 * scene_depth

In [ ]:
# Build the toed in rotation matrix for the novel views
def look_at_opencv(eye, target, up_hint):
    """
    Build a camera-to-world rotation R_c2w (3x3) for an OpenCV-convention camera
    (+x right, +y down, +z forward) placed at `eye`, looking at `target`,
    with `up_hint` indicating which world direction should map to camera-up
    (i.e. opposite of the camera's +y axis).

    Returns R_c2w (3,3) and translation = eye (3,).
    """
    eye    = np.asarray(eye,    dtype=np.float64)
    target = np.asarray(target, dtype=np.float64)
    up_hint = np.asarray(up_hint, dtype=np.float64)

    # Compute the new forward axis towards the target as norm(t - c')
    forward = target - eye
    forward /= np.linalg.norm(forward)

    # Get right-pointing axis
    right = np.cross(up_hint, forward)
    right /= np.linalg.norm(right)

    # down = forward x right completes the right-handed frame.
    down = np.cross(forward, right)
    down /= np.linalg.norm(down)

    R_c2w = np.stack([right, down, forward], axis=1)   # columns are camera axes in world
    return R_c2w, eye


# Compute extrinsics matrix - (4x4) homogeneous transformation 
def c2w_to_w2c(R_c2w, c):
    R_w2c = R_c2w.T
    t_w2c = -R_w2c @ c
    extr = np.zeros((4, 4), dtype=np.float64)
    extr[:3, :3] = R_w2c
    extr[:3,  3] = t_w2c
    extr[ 3,  3] = 1.0
    return extr

up_hint = down_w   # world up = -down_w from earlier

# Four shifts: (label, dx, dy) in camera right/up units.
shifts = [
    ("left",  -delta, 0.0),
    ("right",  delta, 0.0),
    ("up",     0.0,  delta),
    ("down",   0.0, -delta),
]

render_exts = []
for label, dx, dy in shifts:
    c_new = c_w + dx * right_w + dy * up_w
    R_c2w_new, _ = look_at_opencv(c_new, T, up_hint)
    extr_4x4 = c2w_to_w2c(R_c2w_new, c_new)
    render_exts.append(extr_4x4)
render_exts = np.stack(render_exts, axis=0).astype(np.float32)   # (4, 4, 4)

# Reuse the predicted intrinsics for each view.
render_ixts = np.broadcast_to(K[None], (4, 3, 3)).astype(np.float32).copy()
render_hw   = (H, W)

In [ ]:
# Pass 2 — predict the 3d gaussians to generate the novel views
prediction_gs = model.inference(
    image=[str(rgb_path)],
    infer_gs=True,
    render_exts=render_exts,
    render_ixts=render_ixts,
    render_hw=render_hw,
    process_res=560,
    # no export_dir, no export_format
)

In [ ]:
# Razterize the novel views
g = prediction_gs.gaussians
means     = g.means[0].cuda().contiguous()                    # (P, 3)
quats     = g.rotations[0].cuda().contiguous()                # (P, 4)
scales    = g.scales[0].cuda().contiguous()                   # (P, 3)
opacities = g.opacities[0].cuda().contiguous()                # (P,)
# DA3 stores SH as (P, 3, 9); gsplat wants (P, K=9, 3). Transpose last two axes.
colors    = g.harmonics[0].permute(0, 2, 1).cuda().contiguous()  # (P, 9, 3)

sh_degree = 2  # 9 SH coefficients → degree 2

H, W = render_hw
render_exts_t = torch.from_numpy(render_exts).float().cuda()  # (4, 4, 4)
render_ixts_t = torch.from_numpy(render_ixts).float().cuda()  # (4, 3, 3)

rendered = []
for extr, ixt in zip(render_exts_t, render_ixts_t):
    rgb, alpha, _ = rasterization(
        means=means,
        quats=quats,
        scales=scales,
        opacities=opacities,
        colors=colors,
        viewmats=extr.unsqueeze(0),   # (1, 4, 4) w2c
        Ks=ixt.unsqueeze(0),          # (1, 3, 3)
        width=W,
        height=H,
        sh_degree=sh_degree,
    )
    # rgb: (1, H, W, 3) in [0, 1]
    img = (rgb[0].clamp(0, 1).detach().cpu().numpy() * 255).astype(np.uint8)
    rendered.append(img)

# Display inline.
labels = ["left", "right", "up", "down"]
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, img, lab in zip(axes, rendered, labels):
    ax.imshow(img); ax.set_title(lab); ax.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
# Sanity check: render the input view itself.
extr_orig = torch.cat([torch.from_numpy(extr_w2c).float(),
                       torch.tensor([[0,0,0,1.]])], dim=0).cuda()  # (4,4)
ixt_orig  = torch.from_numpy(K).float().cuda()
rgb, _, _ = rasterization(
    means=means, quats=quats, scales=scales, opacities=opacities, colors=colors,
    viewmats=extr_orig.unsqueeze(0), Ks=ixt_orig.unsqueeze(0),
    width=W, height=H, sh_degree=sh_degree,
)
plt.imshow((rgb[0].clamp(0,1).cpu().numpy()*255).astype(np.uint8))
plt.title("sanity: reproduced input view"); plt.axis("off"); plt.show()

In [ ]:
# Rerun the rasterized reproduced view through DA3, then mask by confidence.
# This is a self-consistency probe: where does DA3 think the synthesized image
# looks like a real photo?

import tempfile

# 1. Render the input viewpoint (already did this for sanity; redo cleanly here).
extr_orig_4x4 = np.eye(4, dtype=np.float32)
extr_orig_4x4[:3, :] = extr_w2c   # extr_w2c is (3,4) from Stage 2
extr_t = torch.from_numpy(extr_orig_4x4).float().cuda().unsqueeze(0)   # (1,4,4)
ixt_t  = torch.from_numpy(K).float().cuda().unsqueeze(0)               # (1,3,3)

rgb_render, _, _ = rasterization(
    means=means, quats=quats, scales=scales,
    opacities=opacities, colors=colors,
    viewmats=extr_t, Ks=ixt_t,
    width=W, height=H, sh_degree=sh_degree,
)
reproduced = (rgb_render[0].clamp(0, 1).detach().cpu().numpy() * 255).astype(np.uint8)

# 2. Run DA3 on the reproduced image. The API expects file paths or PIL/np, so
#    we save to a temp file (simplest path through the API).
with tempfile.NamedTemporaryFile(suffix=".png", delete=False) as tmp:
    Image.fromarray(reproduced).save(tmp.name)
    pred_reproduced = model.inference(
        image=[tmp.name],
        infer_gs=False,
        process_res=560,
    )

conf_reproduced = pred_reproduced.conf[0]          # (H', W') — may differ from H,W
depth_reproduced = pred_reproduced.depth[0]

# 3. Threshold at the 50th percentile.
thresh = np.percentile(conf_reproduced, 50)
mask = conf_reproduced >= thresh                   # True where confident

# 4. Apply the mask: keep confident regions, gray out the rest.
H2, W2 = conf_reproduced.shape
# Resize reproduced image to match the conf map if needed (DA3 may have resized internally).
from PIL import Image as PILImage
if (H2, W2) != reproduced.shape[:2]:
    reproduced_resized = np.array(
        PILImage.fromarray(reproduced).resize((W2, H2), PILImage.BILINEAR)
    )
else:
    reproduced_resized = reproduced

masked_rgb = reproduced_resized.copy()
masked_rgb[~mask] = 128   # gray out low-confidence pixels

# 5. Visualize: reproduced | conf heatmap | masked.
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(reproduced_resized); axes[0].set_title("DA3 reproduced view"); axes[0].axis("off")
im = axes[1].imshow(conf_reproduced, cmap="viridis")
axes[1].set_title(f"DA3 conf (thresh @ p50 = {thresh:.3f})"); axes[1].axis("off")
plt.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)
axes[2].imshow(masked_rgb); axes[2].set_title("Masked: confident regions only"); axes[2].axis("off")
plt.tight_layout(); plt.show()

print(f"Confident pixel fraction: {mask.mean():.1%}")

In [ ]:
# Save to disk
from PIL import Image
from pathlib import Path
out = Path("/home/nmeurer/monocular-depth-estimation/outputs/single_image_shifts")
out.mkdir(parents=True, exist_ok=True)
for img, lab in zip(rendered, labels):
    Image.fromarray(img).save(out / f"shift_{lab}.png")

In [ ]:
# Re-run each rasterized novel view through the prediction head,
# then mask out pixels whose confidence is below the 50th percentile.
novel_depth_results = []
for img_np in rendered:
    pred_novel = model.inference(
        image=[Image.fromarray(img_np)],
        process_res=min(H, W),
    )
    depth_novel = pred_novel.depth.squeeze(0)  # (H, W)
    conf_novel  = pred_novel.conf.squeeze(0)   # (H, W)
    conf_thresh  = np.percentile(conf_novel, 50)
    depth_masked = np.where(conf_novel >= conf_thresh, depth_novel, np.nan)
    novel_depth_results.append((depth_novel, conf_novel, depth_masked))
# Visualise: rendered RGB | raw predicted depth | confidence-masked depth
fig, axes = plt.subplots(len(rendered), 3, figsize=(12, 3 * len(rendered)))
col_titles = ["Rendered RGB", "Predicted Depth", "Depth (conf ≥ p50)"]
for row, (img_np, (depth_raw, conf, depth_masked), lab) in enumerate(
    zip(rendered, novel_depth_results, labels)
):
    for ax, col_title in zip(axes[row], col_titles):
        ax.set_title(f"{lab} — {col_title}", fontsize=8)
        ax.axis("off")
    axes[row, 0].imshow(img_np)
    valid = np.isfinite(depth_raw) & (depth_raw > 0)
    lo, hi = np.percentile(depth_raw[valid], [1, 99])
    axes[row, 1].imshow(depth_raw,    cmap="viridis", vmin=lo, vmax=hi)
    axes[row, 2].imshow(depth_masked, cmap="viridis", vmin=lo, vmax=hi)
plt.tight_layout()
plt.show()


In [ ]:
# =============================================================================
# Method B end-to-end: provenance-based confidence masking on novel views.
#
# Inputs assumed in scope from earlier cells:
#   prediction        : first-pass DA3 prediction on the input image (no GS)
#                       prediction.conf[0] -> (H_in, W_in) input-view confidence
#                       prediction.depth[0] -> (H_in, W_in) input-view depth
#   means, quats, scales, opacities, colors, sh_degree : Gaussian tensors on CUDA
#   extr_w2c, K       : input camera (3,4 w2c) and intrinsics (3,3)  -- numpy
#   c_w, right_w, up_w, down_w, forward_w : input camera world-axis vectors
#   scene_depth       : median depth used as scene scale
#   H, W              : processed image dims
#   rgb_path, depth_path : paths to original RGB and GT depth (.npy) for display
# =============================================================================

import numpy as np, torch, tempfile
import matplotlib.pyplot as plt
from PIL import Image
from gsplat import rasterization

USE_TOE_IN = True
DELTA      = 0.05 * scene_depth           # shift magnitude
P_THRESH   = 50.0                         # percentile for the trust mask

# ---- 0. Per-Gaussian confidence (provenance) --------------------------------
# One Gaussian per input pixel of the processed view, row-major.
conf_input = prediction.conf[0]                                          # (H_in, W_in)
P = means.shape[0]
assert conf_input.size == P, f"Conf has {conf_input.size} px but {P} Gaussians."
conf_per_gaussian = torch.from_numpy(conf_input.reshape(-1)).float().cuda()  # (P,)
conf_color = conf_per_gaussian.unsqueeze(-1).expand(-1, 3).contiguous()       # (P,3)

# Per-Gaussian depth in WORLD frame doesn't make sense for splatting; what we
# splat is the per-Gaussian VALUE we want to read out. For a depth pseudo-label
# of the novel view we want camera-space z. We'll compute that per-camera below.

# ---- 1. Build the four toe-in cameras (and reuse the input camera too) ------
def look_at_opencv(eye, target, up_hint):
    f = target - eye; f /= np.linalg.norm(f)
    r = np.cross(up_hint, f); r /= np.linalg.norm(r)
    d = np.cross(f, r);       d /= np.linalg.norm(d)
    return np.stack([r, d, f], axis=1)

def c2w_to_w2c_4x4(R_c2w, c):
    R = R_c2w.T
    M = np.eye(4, dtype=np.float64); M[:3,:3] = R; M[:3,3] = -R @ c
    return M

# Reference up for look-at: world-up estimate (we use -down_w of the input).
up_hint = -down_w        # if your earlier rendering came out flipped, try `down_w`
T_target = c_w + scene_depth * forward_w

shifts = [("left", -DELTA, 0.0), ("right", DELTA, 0.0),
          ("up",    0.0,  DELTA), ("down",  0.0, -DELTA)]

# Include the input view as the first camera so we can also visualise the
# reproduced view + its mask alongside the novel ones.
cameras = []   # list of (label, extr_4x4_w2c_np, K_np)

# Input view
M_in = np.eye(4, dtype=np.float64); M_in[:3,:] = extr_w2c
cameras.append(("input", M_in.astype(np.float32), K.astype(np.float32)))

# Four toe-in shifts
R_c2w_in = extr_w2c[:, :3].T
for label, dx, dy in shifts:
    c_new = c_w + dx * right_w + dy * up_w
    if USE_TOE_IN:
        R_c2w_new = look_at_opencv(c_new, T_target, up_hint)
    else:
        R_c2w_new = R_c2w_in
    extr_4x4 = c2w_to_w2c_4x4(R_c2w_new, c_new).astype(np.float32)
    cameras.append((label, extr_4x4, K.astype(np.float32)))

# ---- 2. Per-camera splatting: RGB, confidence, depth ------------------------
def splat_rgb(extr, K_intr):
    img, alpha, _ = rasterization(
        means=means, quats=quats, scales=scales, opacities=opacities,
        colors=colors,
        viewmats=torch.from_numpy(extr).cuda().unsqueeze(0),
        Ks=torch.from_numpy(K_intr).cuda().unsqueeze(0),
        width=W, height=H, sh_degree=sh_degree,
    )
    return img[0].clamp(0,1), alpha[0,...,0]   # (H,W,3), (H,W)

def splat_scalar(extr, K_intr, per_gauss_scalar_3ch):
    img, alpha, _ = rasterization(
        means=means, quats=quats, scales=scales, opacities=opacities,
        colors=per_gauss_scalar_3ch,
        viewmats=torch.from_numpy(extr).cuda().unsqueeze(0),
        Ks=torch.from_numpy(K_intr).cuda().unsqueeze(0),
        width=W, height=H, sh_degree=None,
    )
    return img[0,...,0], alpha[0,...,0]        # (H,W), (H,W)

def camera_space_depth_per_gaussian(extr_w2c_4x4):
    """z-coordinate of each Gaussian in the given camera's frame."""
    R = torch.from_numpy(extr_w2c_4x4[:3,:3]).float().cuda()
    t = torch.from_numpy(extr_w2c_4x4[:3, 3]).float().cuda()
    z = (means @ R.T)[:, 2] + t[2]              # (P,)
    return z.unsqueeze(-1).expand(-1, 3).contiguous()

results = []   # list of dicts with rgb, depth, trust, mask
for label, extr, K_intr in cameras:
    rgb, alpha_rgb        = splat_rgb(extr, K_intr)
    conf_img, alpha_conf  = splat_scalar(extr, K_intr, conf_color)
    depth_color           = camera_space_depth_per_gaussian(extr)
    depth_img, _          = splat_scalar(extr, K_intr, depth_color)

    # Normalise splatted confidence by coverage so a thinly-covered pixel
    # doesn't look low-conf just because background is zero. Then combine
    # with alpha to penalise thin regions explicitly.
    conf_norm = conf_img / alpha_conf.clamp(min=1e-6)
    trust     = (alpha_conf * conf_norm).detach().cpu().numpy()
    depth_np  = depth_img.detach().cpu().numpy()
    rgb_np    = (rgb.detach().cpu().numpy() * 255).astype(np.uint8)

    thresh = np.percentile(trust, P_THRESH)
    mask   = trust >= thresh

    results.append({"label": label, "rgb": rgb_np, "depth": depth_np,
                    "trust": trust, "mask": mask})

# ---- 3. Display the input image + GT depth + first-pass predicted depth ----
rgb_orig = np.array(Image.open(rgb_path).convert("RGB"))
gt_depth = np.load(depth_path).astype(np.float32)
gt_vis   = np.where(gt_depth > 0, gt_depth, np.nan)
pred_depth = prediction.depth[0]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(rgb_orig);                axes[0].set_title("Input RGB");          axes[0].axis("off")
im1 = axes[1].imshow(gt_vis, cmap="viridis"); axes[1].set_title("GT depth");      axes[1].axis("off")
plt.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)
im2 = axes[2].imshow(pred_depth, cmap="viridis"); axes[2].set_title("DA3 first-pass depth"); axes[2].axis("off")
plt.colorbar(im2, ax=axes[2], fraction=0.046, pad=0.04)
plt.tight_layout(); plt.show()

# ---- 4. Display each view: RGB | splatted depth | masked depth -------------
n = len(results)
fig, axes = plt.subplots(n, 3, figsize=(15, 4*n))
for i, r in enumerate(results):
    axes[i,0].imshow(r["rgb"]);                axes[i,0].set_title(f"{r['label']}: RGB");        axes[i,0].axis("off")

    im_d = axes[i,1].imshow(r["depth"], cmap="viridis")
    axes[i,1].set_title(f"{r['label']}: splatted depth");                                          axes[i,1].axis("off")
    plt.colorbar(im_d, ax=axes[i,1], fraction=0.046, pad=0.04)

    masked_depth = np.where(r["mask"], r["depth"], np.nan)
    im_m = axes[i,2].imshow(masked_depth, cmap="viridis")
    kept = r["mask"].mean() * 100
    axes[i,2].set_title(f"{r['label']}: masked depth (p{P_THRESH:.0f}, kept {kept:.0f}%)")
    axes[i,2].axis("off")
    plt.colorbar(im_m, ax=axes[i,2], fraction=0.046, pad=0.04)

plt.tight_layout(); plt.show()